In [1]:
import os
import dotenv

dotenv.load_dotenv()

if "DATA_DIR" not in os.environ:
    
    raise Exception("Please set the DATA_DIR environment variable to where you want to save your dataset")

DATA_DIR = os.environ["DATA_DIR"]
assert DATA_DIR, "Please set the DATA_DIR environment variable to your data directory"

OUPTUT_DIR = os.path.join(DATA_DIR, "vsr") 


In [2]:
from datasets import load_dataset


vsr_dataset = load_dataset("cambridgeltl/vsr_zeroshot")

In [3]:
vsr_dataset['train'][0]

{'image': '000000558388.jpg',
 'image_link': 'http://images.cocodataset.org/train2017/000000558388.jpg',
 'caption': 'The cake is next to the person.',
 'label': 1,
 'relation': 'next to',
 'subj': 'cake',
 'obj': 'person',
 'annotator_id': 35,
 'vote_true_validator_id': '[2, 67, 20]',
 'vote_false_validator_id': '[]'}

In [6]:
import os
import requests
from datasets import load_dataset, Dataset
from tqdm import tqdm

# Output directories
images_dir = os.path.join(OUPTUT_DIR, "images")
os.makedirs(images_dir, exist_ok=True)

In [7]:
# download image and return local path
def download_and_replace_image(example):
    image_url = example["image_link"]
    image_name = example["image"]
    local_path = os.path.join(images_dir, image_name)
    
    # Download only if not already present
    if not os.path.exists(local_path):
        try:
            response = requests.get(image_url, timeout=10)
            response.raise_for_status()
            with open(local_path, "wb") as f:
                f.write(response.content)
        except Exception as e:
            print(f"Failed to download {image_url}: {e}")
            local_path = None  # or set to empty string if needed

    # Replace the 'image' field with the local file path
    return {
        "image_path": local_path,
        "caption": example["caption"],
        "label": example["label"],
        "relation": example["relation"],
        "subj": example["subj"],
        "obj": example["obj"]
    }

In [ ]:
# Map the dataset with image downloading
vsr_dataset = vsr_dataset.map(download_and_replace_image)

Map:   0%|          | 0/3489 [00:00<?, ? examples/s]

In [ ]:
# Remove unused columns (optional, in case you want a clean dataset)
vsr_dataset = vsr_dataset.remove_columns([col for col in vsr_dataset.column_names['train'] if col not in ["image_path", "caption", "label", "relation", "subj", "obj"]])

In [ ]:
# Sample
vsr_dataset['train'][0]

{'caption': 'The cake is next to the person.',
 'label': 1,
 'relation': 'next to',
 'subj': 'cake',
 'obj': 'person',
 'image_path': '/scratch/izar/vanousek/vlm_r1/data/images/vsr/images/000000558388.jpg'}

In [ ]:
# Save the dataset to disk
vsr_dataset.save_to_disk(output_dir) # # change this to your path where you want to save the dataset

Saving the dataset (0/1 shards):   0%|          | 0/3489 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/340 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1222 [00:00<?, ? examples/s]